# CellXGene Tissue Hierarchy with Official API

This notebook demonstrates how to use the `cellxgene-ontology-guide` package to query tissue-organ-system hierarchies.

## Installation
```bash
pip install cellxgene-ontology-guide
```

In [1]:
from cellxgene_ontology_guide.ontology_parser import OntologyParser
from cellxgene_ontology_guide.curated_ontology_term_lists import (
    CuratedOntologyTermList,
    get_curated_ontology_term_list
)
import pandas as pd

## Initialize the Ontology Parser

In [2]:
# Initialize ontology parser (loads UBERON and other ontologies)
ontology = OntologyParser()

# Get CellXGene's curated lists
tissue_list = get_curated_ontology_term_list(CuratedOntologyTermList.TISSUE_GENERAL)
organ_list = get_curated_ontology_term_list(CuratedOntologyTermList.ORGAN)
system_list = get_curated_ontology_term_list(CuratedOntologyTermList.SYSTEM)

organ_set = set(organ_list)
system_set = set(system_list)

print(f"CellXGene curated terms:")
print(f"  {len(tissue_list)} tissues")
print(f"  {len(organ_list)} organs")
print(f"  {len(system_list)} systems")

CellXGene curated terms:
  81 tissues
  28 organs
  17 systems


## View Curated Lists

In [3]:
# Show all curated organs
print("CellXGene Canonical Organs:")
for organ_id in organ_list:
    print(f"  {ontology.get_term_label(organ_id):30} {organ_id}")

CellXGene Canonical Organs:
  ovary                          UBERON:0000992
  lymph node                     UBERON:0000029
  lung                           UBERON:0002048
  gallbladder                    UBERON:0002110
  esophagus                      UBERON:0001043
  fallopian tube                 UBERON:0003889
  bladder organ                  UBERON:0018707
  blood                          UBERON:0000178
  bone marrow                    UBERON:0002371
  brain                          UBERON:0000955
  breast                         UBERON:0000310
  eye                            UBERON:0000970
  heart                          UBERON:0000948
  intestine                      UBERON:0000160
  kidney                         UBERON:0002113
  liver                          UBERON:0002107
  nose                           UBERON:0000004
  pancreas                       UBERON:0001264
  placenta                       UBERON:0001987
  skin of body                   UBERON:0002097
  spinal cor

In [4]:
# Show all curated systems
print("CellXGene Canonical Systems:")
for system_id in system_list:
    print(f"  {ontology.get_term_label(system_id):30} {system_id}")

CellXGene Canonical Systems:
  central nervous system         UBERON:0001017
  cardiovascular system          UBERON:0004535
  circulatory system             UBERON:0001009
  digestive system               UBERON:0001007
  embryo                         UBERON:0000922
  endocrine system               UBERON:0000949
  exocrine system                UBERON:0002330
  hematopoietic system           UBERON:0002390
  immune system                  UBERON:0002405
  musculature of body            UBERON:0000383
  nervous system                 UBERON:0001016
  peripheral nervous system      UBERON:0000010
  renal system                   UBERON:0001008
  reproductive system            UBERON:0000990
  respiratory system             UBERON:0001004
  sensory system                 UBERON:0001032
  skeletal system                UBERON:0001434


## Helper Function: Get Tissue Hierarchy

In [5]:
def get_tissue_hierarchy(tissue_id):
    """
    Get the organ and system classifications for a tissue.
    
    Returns:
        dict with tissue_label, organs (list of tuples), systems (list of tuples)
    """
    # Get all ancestors (including the tissue itself)
    ancestors = ontology.get_term_ancestors(tissue_id, include_self=True)
    
    # Find which curated organs and systems are ancestors
    organs = [(oid, ontology.get_term_label(oid)) for oid in ancestors if oid in organ_set]
    systems = [(sid, ontology.get_term_label(sid)) for sid in ancestors if sid in system_set]
    
    return {
        'tissue_id': tissue_id,
        'tissue_label': ontology.get_term_label(tissue_id),
        'organs': organs,
        'systems': systems
    }

# Test it
result = get_tissue_hierarchy('UBERON:0000178')  # blood
print(f"Tissue: {result['tissue_label']} ({result['tissue_id']})")
print(f"Organs: {result['organs']}")
print(f"Systems: {result['systems']}")

Tissue: blood (UBERON:0000178)
Organs: [('UBERON:0000178', 'blood')]
Systems: [('UBERON:0002390', 'hematopoietic system')]


## Example Queries

In [6]:
# Query several interesting tissues
test_tissues = [
    ('UBERON:0000178', 'blood'),
    ('UBERON:0002048', 'lung'),
    ('UBERON:0002369', 'adrenal gland'),
    ('UBERON:0002082', 'cardiac ventricle'),
    ('UBERON:0001225', 'cortex of kidney'),
]

for tissue_id, expected_name in test_tissues:
    result = get_tissue_hierarchy(tissue_id)
    print(f"\n{result['tissue_label']} ({tissue_id}):")
    print(f"  Organs: {[label for _, label in result['organs']]}")
    print(f"  Systems: {[label for _, label in result['systems']]}")


blood (UBERON:0000178):
  Organs: ['blood']
  Systems: ['hematopoietic system']

lung (UBERON:0002048):
  Organs: ['lung']
  Systems: ['respiratory system']

adrenal gland (UBERON:0002369):
  Organs: []
  Systems: ['endocrine system']

cardiac ventricle (UBERON:0002082):
  Organs: ['heart']
  Systems: ['cardiovascular system', 'circulatory system']

cortex of kidney (UBERON:0001225):
  Organs: ['kidney']
  Systems: ['renal system']


## Build Complete Mapping for All Curated Tissues

This creates a dataframe with all CellXGene curated tissues and their organ/system assignments.

In [12]:
# Build mapping for all curated tissues
rows = []

for tissue_id in tissue_list:
    result = get_tissue_hierarchy(tissue_id)
    
    # If tissue has multiple organs/systems, create one row per combination
    organs = result['organs'] if result['organs'] else [(None, None)]
    systems = result['systems'] if result['systems'] else [(None, None)]
    
    for organ_id, organ_label in organs:
        for system_id, system_label in systems:
            rows.append({
                'tissue_label': result['tissue_label'],
                'tissue_uberon_id': tissue_id,
                'organ_label': organ_label,
                'organ_uberon_id': organ_id,
                'system_label': system_label,
                'system_uberon_id': system_id,
            })

df = pd.DataFrame(rows)
df = df.sort_values(['system_label', 'organ_label', 'tissue_label'], na_position='last')

print(f"Total mappings: {len(df)}")
print(f"Unique tissues: {df['tissue_uberon_id'].nunique()}")
df.head(20)

Total mappings: 101
Unique tissues: 81


,tissue_label,tissue_uberon_id,organ_label,organ_uberon_id,system_label,system_uberon_id
16,heart,UBERON:0000948,heart,UBERON:0000948,cardiovascular system,UBERON:0004535
70,blood vasculature,UBERON:0004537,None,None,cardiovascular system,UBERON:0004535
85,cardiovascular system,UBERON:0004535,None,None,cardiovascular system,UBERON:0004535
53,lymph vasculature,UBERON:0004536,None,None,cardiovascular system,UBERON:0004535
62,vasculature,UBERON:0002049,None,None,cardiovascular system,UBERON:0004535
11,brain,UBERON:0000955,brain,UBERON:0000955,central nervous system,UBERON:0001017
13,spinal cord,UBERON:0002240,spinal cord,UBERON:0002240,central nervous system,UBERON:0001017
76,central nervous system,UBERON:0001017,None,None,central nervous system,UBERON:0001017
17,heart,UBERON:0000948,heart,UBERON:0000948,circulatory system,UBERON:0001009
71,blood vasculature,UBERON:0004537,None,None,circulatory system,UBERON:0001009


In [19]:
import duckdb
duckdb.sql("SELECT distinct substring(organ_uberon_id, 1, 2) FROM df")

┌─────────────────────────────────────────┐
│ main."substring"(organ_uberon_id, 1, 2) │
│                 varchar                 │
├─────────────────────────────────────────┤
│ NULL                                    │
│ UB                                      │
└─────────────────────────────────────────┘

## Analysis: Coverage Statistics

In [9]:
print("Coverage Statistics:\n")
print(f"Tissues with both organ and system: {(df['organ_uberon_id'].notna() & df['system_uberon_id'].notna()).sum()}")
print(f"Tissues with only system (no organ): {(df['organ_uberon_id'].isna() & df['system_uberon_id'].notna()).sum()}")
print(f"Tissues with only organ (no system): {(df['organ_uberon_id'].notna() & df['system_uberon_id'].isna()).sum()}")
print(f"Tissues with neither: {(df['organ_uberon_id'].isna() & df['system_uberon_id'].isna()).sum()}")

Coverage Statistics:

Tissues with both organ and system: 39
Tissues with only system (no organ): 41
Tissues with only organ (no system): 12
Tissues with neither: 9


## Working with Your Dataset

Example: Add organ and system columns to a dataset with tissue_ontology_term_id

In [ ]:
# Example: Your dataset with tissue IDs
example_data = pd.DataFrame({
    'cell_id': ['cell1', 'cell2', 'cell3', 'cell4'],
    'tissue_ontology_term_id': ['UBERON:0000178', 'UBERON:0002048', 'UBERON:0002113', 'UBERON:0002369']
})

print("Original data:")
print(example_data)

# Add hierarchy information
def add_hierarchy_info(tissue_id):
    result = get_tissue_hierarchy(tissue_id)
    # Return primary (first) organ and system
    organ = result['organs'][0] if result['organs'] else (None, None)
    system = result['systems'][0] if result['systems'] else (None, None)
    return pd.Series({
        'tissue_label': result['tissue_label'],
        'organ_label': organ[1],
        'organ_uberon_id': organ[0],
        'system_label': system[1],
        'system_uberon_id': system[0]
    })

# Apply to dataset
hierarchy_info = example_data['tissue_ontology_term_id'].apply(add_hierarchy_info)
result_df = pd.concat([example_data, hierarchy_info], axis=1)

print("\nWith hierarchy info:")
print(result_df)

## Advanced: Explore the Full Ontology

In [10]:
# Get all ancestors of a tissue with distances
tissue_id = 'UBERON:0002369'  # adrenal gland
ancestors_with_dist = ontology.get_term_ancestors_with_distances(tissue_id)

print(f"Ancestors of {ontology.get_term_label(tissue_id)}:\n")
for ancestor_id, distance in sorted(ancestors_with_dist.items(), key=lambda x: x[1])[:10]:
    label = ontology.get_term_label(ancestor_id)
    in_organ = '(ORGAN)' if ancestor_id in organ_set else ''
    in_system = '(SYSTEM)' if ancestor_id in system_set else ''
    print(f"  Distance {distance}: {label:40} {ancestor_id} {in_organ} {in_system}")

Ancestors of adrenal gland:

  Distance 1: abdomen element                          UBERON:0005172  
  Distance 1: adrenal/interrenal gland                 UBERON:0006858  
  Distance 1: structure with developmental contribution from neural crest UBERON:0010314  
  Distance 1: lateral structure                        UBERON:0015212  
  Distance 1: abdomen                                  UBERON:0000916  
  Distance 1: endocrine system                         UBERON:0000949  (SYSTEM)
  Distance 2: abdominal segment element                UBERON:0005173  
  Distance 2: endocrine gland                          UBERON:0002368  
  Distance 2: mesoderm-derived structure               UBERON:0004120  
  Distance 2: chromaffin system                        UBERON:0010074  


In [ ]:
# Get children (descendants one level down)
organ_id = 'UBERON:0002113'  # kidney
children = ontology.get_term_children(organ_id)

print(f"Direct children of {ontology.get_term_label(organ_id)}:\n")
for child_id in sorted(children)[:10]:
    print(f"  {ontology.get_term_label(child_id):40} {child_id}")

## Export Full Mapping (Optional)

If you need a static file for reference:

In [11]:
# Save to CSV
output_path = '../data/cellxgene_tissue_system_full.csv'
df.to_csv(output_path, index=False)
print(f"Saved mapping to {output_path}")
print(f"Total rows: {len(df)}")

Saved mapping to ../data/cellxgene_tissue_system_full.csv
Total rows: 101
